In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

artifacts_dir = Path("../artifacts")

train_df = pd.read_parquet(
    artifacts_dir / "05_train_features.parquet"
)

val_df = pd.read_parquet(
    artifacts_dir / "05_validation_features.parquet"
)

test_df = pd.read_parquet(
    artifacts_dir / "05_test_features.parquet"
)

target = "late_delivery"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val = val_df.drop(columns=[target])
y_val = val_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (67533, 7361)
Validation: (14471, 7361)
Test: (14472, 7361)


In [2]:
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)

print("Baseline validation results")
print(
    classification_report(
        y_val,
        baseline_pred,
        target_names=["On-time", "Late"],
        zero_division=0
    )
)

Baseline validation results
              precision    recall  f1-score   support

     On-time       0.92      1.00      0.96     13297
        Late       0.00      0.00      0.00      1174

    accuracy                           0.92     14471
   macro avg       0.46      0.50      0.48     14471
weighted avg       0.84      0.92      0.88     14471



In [3]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [4]:
val_pred = model.predict(X_val)
val_prob = model.predict_proba(X_val)[:, 1]

print("Validation results")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["On-time", "Late"],
        zero_division=0
    )
)

Validation results
              precision    recall  f1-score   support

     On-time       0.94      0.66      0.77     13297
        Late       0.12      0.55      0.20      1174

    accuracy                           0.65     14471
   macro avg       0.53      0.60      0.49     14471
weighted avg       0.88      0.65      0.73     14471



In [5]:
val_metrics = {
    "accuracy": accuracy_score(y_val, val_pred),
    "precision": precision_score(
        y_val, val_pred, zero_division=0
    ),
    "recall": recall_score(
        y_val, val_pred, zero_division=0
    ),
    "f1": f1_score(
        y_val, val_pred, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_val, val_prob
    ),
    "pr_auc": average_precision_score(
        y_val, val_prob
    )
}

for metric, value in val_metrics.items():
    print(f"{metric}: {value:.4f}")

accuracy: 0.6479
precision: 0.1238
recall: 0.5494
f1: 0.2020
roc_auc: 0.6302
pr_auc: 0.1312


In [6]:
cm = confusion_matrix(y_val, val_pred)

print("Validation confusion matrix:")
print(cm)

print("\nRows = actual")
print("Columns = predicted")
print("\n[[TN, FP],")
print(" [FN, TP]]")

Validation confusion matrix:
[[8731 4566]
 [ 529  645]]

Rows = actual
Columns = predicted

[[TN, FP],
 [FN, TP]]


In [7]:
results = []

for C in [0.01, 0.1, 1.0, 10.0]:
    tuned_model = LogisticRegression(
        C=C,
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    tuned_model.fit(X_train, y_train)

    pred = tuned_model.predict(X_val)
    prob = tuned_model.predict_proba(X_val)[:, 1]

    results.append({
        "C": C,
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_val, prob
        ),
        "pr_auc": average_precision_score(
            y_val, prob
        )
    })

tuning_results = pd.DataFrame(results)

display(
    tuning_results.sort_values(
        "f1",
        ascending=False
    )
)

,C,precision,recall,f1,roc_auc,pr_auc
1,0.10,0.126808,0.545145,0.205755,0.644063,0.135846
0,0.01,0.125452,0.531516,0.202993,0.645078,0.134187
2,1.00,0.123777,0.549404,0.202036,0.630226,0.131242
3,10.00,0.122529,0.559625,0.201040,0.624244,0.126170


In [8]:
best_row = tuning_results.loc[
    tuning_results["f1"].idxmax()
]

best_C = best_row["C"]

print("Selected C:", best_C)

Selected C: 0.1


In [9]:
final_model = LogisticRegression(
    C=best_C,
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_train, y_train)

print("Final model trained.")

Final model trained.


In [10]:
test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(
        y_test, test_pred, zero_division=0
    ),
    "recall": recall_score(
        y_test, test_pred, zero_division=0
    ),
    "f1": f1_score(
        y_test, test_pred, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test, test_prob
    ),
    "pr_auc": average_precision_score(
        y_test, test_prob
    )
}

print("FINAL TEST RESULTS\n")

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

FINAL TEST RESULTS

accuracy: 0.6666
precision: 0.1311
recall: 0.5528
f1: 0.2120
roc_auc: 0.6561
pr_auc: 0.1435


In [11]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["On-time", "Late"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

     On-time       0.94      0.68      0.79     13298
        Late       0.13      0.55      0.21      1174

    accuracy                           0.67     14472
   macro avg       0.54      0.61      0.50     14472
weighted avg       0.88      0.67      0.74     14472



In [12]:
print("Final test confusion matrix:")
print(confusion_matrix(y_test, test_pred))

Final test confusion matrix:
[[8998 4300]
 [ 525  649]]


In [13]:
joblib.dump(
    final_model,
    artifacts_dir / "06_logistic_regression.joblib"
)

print("Saved: ../artifacts/06_logistic_regression.joblib")

Saved: ../artifacts/06_logistic_regression.joblib


In [14]:
results_summary = pd.DataFrame([
    {
        "model": "DummyClassifier",
        "dataset": "validation",
        "accuracy": accuracy_score(y_val, baseline_pred),
        "precision": precision_score(
            y_val, baseline_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, baseline_pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, baseline_pred, zero_division=0
        )
    },
    {
        "model": "LogisticRegression",
        "dataset": "test",
        **test_metrics
    }
])

results_summary.to_csv(
    artifacts_dir / "06_results_summary.csv",
    index=False
)

display(results_summary)

,model,dataset,accuracy,precision,recall,f1,roc_auc,pr_auc
0,DummyClassifier,validation,0.918872,0.000000,0.000000,0.000000,NaN,NaN
1,LogisticRegression,test,0.666598,0.131138,0.552811,0.211988,0.65608,0.143451
